# Biomedical NER Fine-Tuning With BioBERT, LoRA, and QLoRA

This notebook trains and evaluates biomedical named entity recognition models on the BC5CDR dataset. It compares a raw BioBERT baseline against parameter-efficient LoRA and QLoRA adapters.

What this notebook does:

- Clones the project repository into a Kaggle runtime.
- Installs the training and evaluation dependencies.
- Checks GPU availability and memory.
- Evaluates the base BioBERT model without adapters.
- Trains and evaluates LoRA and QLoRA adapters.
- Builds a comparison table and plots for portfolio reporting.
- Includes optional Hugging Face Hub upload code, commented out by default.

Repository: https://github.com/LNS-Files/biomedical-ner-finetune

## Setup: Clone Repository And Install Dependencies

Kaggle notebooks start from a clean runtime. This step clones the project repository, moves into it, and installs the exact dependencies pinned in `requirements.txt`. Replace the placeholder repository URL with your final GitHub URL if needed.

In [ ]:
REPO_URL = "https://github.com/LNS-Files/biomedical-ner-finetune"
REPO_DIR = "/kaggle/working/biomedical-ner-finetune"

!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -r requirements.txt

## Weights & Biases Login

The training config reports to Weights & Biases by default. Run this cell and paste your W&B API key when prompted. If you prefer not to use W&B, set `WANDB_MODE=offline` or change `report_to` in the training config.

In [ ]:
import wandb

wandb.login()

## GPU Check

BioBERT fine-tuning is much faster on a GPU. This cell verifies CUDA availability and prints the GPU name and total memory so the run is easy to document and debug.

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if cuda_available:
    device_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_index)
    total_memory_gb = props.total_memory / (1024 ** 3)
    print(f"GPU: {props.name}")
    print(f"Total memory: {total_memory_gb:.2f} GB")
else:
    print("No CUDA GPU detected. Enable a GPU accelerator in Kaggle before training.")

## Baseline Evaluation

Before training adapters, evaluate the raw BioBERT token classification model. This gives a baseline row for the final comparison table.

In [ ]:
from pathlib import Path
import json
import re
import subprocess

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

def run_command(command, log_path):
    completed = subprocess.run(command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path = Path(log_path)
    log_path.write_text(completed.stdout, encoding="utf-8")
    print(completed.stdout)
    completed.check_returncode()
    return completed.stdout

def parse_eval_metrics(stdout):
    patterns = {
        "test_f1": r"Overall\s+F1\s*:\s*([0-9.]+)",
        "test_precision": r"Overall\s+Precision\s*:\s*([0-9.]+)",
        "test_recall": r"Overall\s+Recall\s*:\s*([0-9.]+)",
    }
    metrics = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, stdout)
        if not match:
            raise ValueError(f"Could not parse {key} from evaluation output")
        metrics[key] = float(match.group(1))
    return metrics

baseline_stdout = run_command("python -m src.evaluate --no-adapter", RESULTS_DIR / "baseline_eval.log")
baseline_metrics = parse_eval_metrics(baseline_stdout)
(RESULTS_DIR / "baseline_metrics.json").write_text(json.dumps(baseline_metrics, indent=2), encoding="utf-8")
baseline_metrics

## LoRA Training

This step trains a LoRA adapter on top of BioBERT. Only the adapter weights and token classification head are trained, which keeps the saved artifact small compared with a full model checkpoint.

In [ ]:
lora_train_stdout = run_command("python -m src.train", RESULTS_DIR / "lora_train.log")

## LoRA Evaluation

After training, reload the saved LoRA adapter and evaluate it on the BC5CDR test split. The parsed metrics are saved to JSON for the comparison table.

In [ ]:
lora_stdout = run_command(
    "python -m src.evaluate --adapter-path results/lora-biobert-bc5cdr",
    RESULTS_DIR / "lora_eval.log",
)
lora_metrics = parse_eval_metrics(lora_stdout)
(RESULTS_DIR / "lora_metrics.json").write_text(json.dumps(lora_metrics, indent=2), encoding="utf-8")
lora_metrics

## QLoRA Training

QLoRA loads the frozen base model in 4-bit quantized form while training LoRA adapters. This is useful when GPU memory is limited, especially on free or shared Kaggle accelerators.

In [ ]:
qlora_train_stdout = run_command("python -m src.train --qlora", RESULTS_DIR / "qlora_train.log")

## QLoRA Evaluation

Evaluate the QLoRA adapter with the matching `--qlora` flag so the base model is loaded using the same quantized setup used during training.

In [ ]:
qlora_stdout = run_command(
    "python -m src.evaluate --adapter-path results/qlora-biobert-bc5cdr --qlora",
    RESULTS_DIR / "qlora_eval.log",
)
qlora_metrics = parse_eval_metrics(qlora_stdout)
(RESULTS_DIR / "qlora_metrics.json").write_text(json.dumps(qlora_metrics, indent=2), encoding="utf-8")
qlora_metrics

## Compare Metrics

Load the three metrics JSON files, combine them into a single pandas DataFrame, save the table as `results/comparison.csv`, and render it inline for GitHub/Kaggle viewers.

In [ ]:
import pandas as pd

metric_files = {
    "Baseline BioBERT": RESULTS_DIR / "baseline_metrics.json",
    "LoRA BioBERT": RESULTS_DIR / "lora_metrics.json",
    "QLoRA BioBERT": RESULTS_DIR / "qlora_metrics.json",
}

rows = []
for model_name, path in metric_files.items():
    metrics = json.loads(path.read_text(encoding="utf-8"))
    rows.append({"model": model_name, **metrics})

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df[["model", "test_f1", "test_precision", "test_recall"]]
comparison_df.to_csv(RESULTS_DIR / "comparison.csv", index=False)
comparison_df

## Plots

The final portfolio artifacts are a model F1 bar chart and training loss curves. The chart summarizes test performance, while the loss curves show whether each adapter converged cleanly.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(comparison_df["model"], comparison_df["test_f1"], color=["#6B7280", "#2563EB", "#059669"])
ax.set_title("BC5CDR Test F1 Comparison")
ax.set_ylabel("Entity-level F1")
ax.set_ylim(0, max(1.0, comparison_df["test_f1"].max() * 1.15))
ax.bar_label(bars, fmt="%.4f", padding=3)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "f1_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

def load_loss_history(trainer_state_path):
    path = Path(trainer_state_path)
    if not path.exists():
        return pd.DataFrame(columns=["step", "loss"])
    state = json.loads(path.read_text(encoding="utf-8"))
    rows = [
        {"step": entry["step"], "loss": entry["loss"]}
        for entry in state.get("log_history", [])
        if "loss" in entry and "step" in entry
    ]
    return pd.DataFrame(rows)

loss_sources = {
    "LoRA": RESULTS_DIR / "lora-biobert-bc5cdr" / "trainer_state.json",
    "QLoRA": RESULTS_DIR / "qlora-biobert-bc5cdr" / "trainer_state.json",
}

fig, ax = plt.subplots(figsize=(8, 5))
for label, state_path in loss_sources.items():
    loss_df = load_loss_history(state_path)
    if loss_df.empty:
        print(f"No loss history found for {label}: {state_path}")
        continue
    ax.plot(loss_df["step"], loss_df["loss"], marker="o", linewidth=2, label=label)

ax.set_title("Training Loss Curves")
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "loss_curves.png", dpi=200, bbox_inches="tight")
plt.show()

## Optional Hugging Face Hub Upload

The code below is commented out so the notebook never uploads artifacts accidentally. When you are ready, uncomment it, replace the repository IDs, and run the cell to publish the adapter folders.

In [ ]:
# from huggingface_hub import HfApi, login
#
# login()  # Paste your Hugging Face token when prompted.
#
# api = HfApi()
#
# api.upload_folder(
#     repo_id="YOUR_USERNAME/biobert-bc5cdr-lora-adapter",
#     folder_path="results/lora-biobert-bc5cdr",
#     repo_type="model",
# )
#
# api.upload_folder(
#     repo_id="YOUR_USERNAME/biobert-bc5cdr-qlora-adapter",
#     folder_path="results/qlora-biobert-bc5cdr",
#     repo_type="model",
# )